# RAG Pipeline -- Retrieval Augmented Generation

## What Is RAG?

**Retrieval Augmented Generation (RAG)** is a technique that grounds LLM responses in external knowledge by retrieving relevant documents before generating an answer. Instead of relying solely on what the model memorized during training, RAG fetches up-to-date, domain-specific context at query time.

The core insight: LLMs are excellent at *reasoning over* provided text, but they hallucinate when asked to recall facts from memory. RAG turns the LLM from an unreliable encyclopedia into a reliable *reading comprehension* engine.

## The RAG Pipeline

```
User Query
    |
    v
[1. RETRIEVE] -- Find the top-K most relevant chunks from the vector store
    |
    v
[2. AUGMENT]  -- Inject retrieved chunks into the LLM prompt as context
    |
    v
[3. GENERATE] -- LLM produces an answer grounded in the retrieved context
    |
    v
Grounded Answer (with source citations)
```

## What You Will Learn

This notebook demonstrates:
1. How the `HybridRetriever` queries multiple vector stores and normalizes results.
2. How **Reciprocal Rank Fusion (RRF)** re-ranking combines results from different stores.
3. How the full pipeline wires together: load, chunk, embed, store, retrieve, generate.

## Prerequisites

* Python 3.10+
* The `agentexplorr` package installed
* No external services needed -- we use mock stores to demonstrate the retrieval logic.

In [ ]:
from dataclasses import dataclass, field
from typing import Any

from agentexplorr.rag.retriever import HybridRetriever, RetrievalResult

print("Imports successful!")
print(f"  HybridRetriever:  {HybridRetriever}")
print(f"  RetrievalResult:  {RetrievalResult}")
print(f"\nValid re-ranking strategies: {HybridRetriever.VALID_STRATEGIES}")

## The Full RAG Flow

Here is how the `RAGPipeline` class orchestrates all the components we have built:

```
[1. LOAD]      DocumentProcessor reads files (PDF, Markdown, Text, HTML, CSV)
                        |
                        v
[2. CHUNK]     Chunker splits each Document into focused Chunk objects
               (FixedSizeChunker, RecursiveChunker, or SemanticChunker)
                        |
                        v
[3. EMBED]     EmbeddingModel converts chunk text into 384-dim vectors
               (all-MiniLM-L6-v2 by default)
                        |
                        v
[4. STORE]     VectorStore indexes the vectors for fast similarity search
               (ChromaDB or FAISS)
                        |
                        v
[5. RETRIEVE]  HybridRetriever queries one or more stores, deduplicates,
               and re-ranks results (by score or RRF)
                        |
                        v
[6. GENERATE]  LLM receives the retrieved context + user question and
               produces a grounded answer (via Ollama / ChatOllama)
```

Steps 1-4 happen once during **ingestion** (offline, batch).
Steps 5-6 happen at **query time** (online, per-question).

Let's focus on step 5 -- the retrieval and re-ranking logic -- since it is the part we can demonstrate without an embedding model or LLM server.

In [ ]:
# --- Build a mock vector store to demonstrate HybridRetriever without embeddings ---

@dataclass
class MockSearchResult:
    """Mimics the SearchResult from ChromaDB/FAISS stores."""
    chunk_id: str = ""
    text: str = ""
    score: float = 0.0
    metadata: dict = field(default_factory=dict)


class MockVectorStore:
    """A fake vector store that returns pre-configured results.

    This lets us demonstrate retriever behavior without needing
    actual embeddings or a running database.
    """
    def __init__(self, name: str, results: list[MockSearchResult]):
        self.name = name
        self._results = results

    def search(self, query: str, top_k: int = 5) -> list[MockSearchResult]:
        """Return pre-configured results (ignoring the actual query)."""
        return self._results[:top_k]


# --- Create two mock stores with overlapping and unique results ---
# Imagine "Store A" is ChromaDB and "Store B" is FAISS.
# They indexed the same documents but found different top results.

store_a_results = [
    MockSearchResult(chunk_id="chunk_attention", text="The key innovation is the self-attention mechanism.", score=0.95, metadata={"source": "transformers.md"}),
    MockSearchResult(chunk_id="chunk_bert", text="BERT uses bidirectional Transformers for pre-training.", score=0.82, metadata={"source": "bert.md"}),
    MockSearchResult(chunk_id="chunk_gpt", text="GPT models use autoregressive Transformers.", score=0.78, metadata={"source": "gpt.md"}),
]

store_b_results = [
    MockSearchResult(chunk_id="chunk_attention", text="The key innovation is the self-attention mechanism.", score=0.91, metadata={"source": "transformers.md"}),
    MockSearchResult(chunk_id="chunk_vit", text="Vision Transformers apply attention to image patches.", score=0.85, metadata={"source": "vit.md"}),
    MockSearchResult(chunk_id="chunk_bert", text="BERT uses bidirectional Transformers for pre-training.", score=0.75, metadata={"source": "bert.md"}),
]

store_a = MockVectorStore("chroma", store_a_results)
store_b = MockVectorStore("faiss", store_b_results)

# --- Create a HybridRetriever with score-based ranking ---
retriever = HybridRetriever(
    stores={"chroma": store_a, "faiss": store_b},
    strategy="score",
)

print(f"Retriever: {retriever}")
print(f"Stores: chroma ({len(store_a_results)} results), faiss ({len(store_b_results)} results)\n")

# --- Search and display results ---
results = retriever.search("What is attention?", top_k=5)

print(f"Score-based re-ranking returned {len(results)} results:\n")
print(f"{'Rank':<6} {'Score':>7} {'Store':<10} {'Chunk ID':<20} {'Text'}")
print("-" * 90)
for rank, r in enumerate(results, 1):
    print(f"{rank:<6} {r.score:>7.4f} {r.source_store:<10} {r.chunk_id:<20} {r.text[:50]}")

## Reciprocal Rank Fusion (RRF) Re-Ranking

Score-based ranking has a subtle problem: **scores from different stores are not directly comparable.** A score of 0.85 from ChromaDB (cosine distance) may not mean the same thing as 0.85 from FAISS (L2 distance converted to similarity). Sorting by raw score can produce misleading rankings.

**Reciprocal Rank Fusion (RRF)** solves this by ignoring raw scores entirely and focusing on **rank position**:

```
RRF_score(chunk) = SUM over each store:  1 / (k + rank_in_store)
```

where `k` is a smoothing constant (default 60) and `rank` is 1-indexed.

### Why RRF Works

* A chunk ranked #1 in **both** stores gets:  `1/(60+1) + 1/(60+1) = 0.0328`
* A chunk ranked #1 in **one** store only gets:  `1/(60+1) = 0.0164`
* Result: chunks that **multiple stores agree on** float to the top.

### Example

| Chunk | Rank in ChromaDB | Rank in FAISS | RRF Score |
|---|---|---|---|
| `chunk_attention` | #1 | #1 | 1/61 + 1/61 = 0.0328 |
| `chunk_bert` | #2 | #3 | 1/62 + 1/63 = 0.0320 |
| `chunk_vit` | (not found) | #2 | 0 + 1/62 = 0.0161 |
| `chunk_gpt` | #3 | (not found) | 1/63 + 0 = 0.0159 |

Notice that `chunk_bert` (found by both stores) ranks higher than `chunk_vit` (found by only one), even though `chunk_vit` had a higher raw score in FAISS. This is the power of ensemble agreement.

In [ ]:
# --- Now use RRF re-ranking with the same two mock stores ---

rrf_retriever = HybridRetriever(
    stores={"chroma": store_a, "faiss": store_b},
    strategy="rrf",
    rrf_k=60,  # The smoothing constant (default)
)

rrf_results = rrf_retriever.search("What is attention?", top_k=5)

print("RRF re-ranking results:\n")
print(f"{'Rank':<6} {'RRF Score':>10} {'Store':<10} {'Chunk ID':<20} {'Text'}")
print("-" * 90)
for rank, r in enumerate(rrf_results, 1):
    print(f"{rank:<6} {r.score:>10.6f} {r.source_store:<10} {r.chunk_id:<20} {r.text[:50]}")

# --- Compare the two strategies side by side ---
print("\n" + "=" * 70)
print("COMPARISON: Score-based vs RRF ranking")
print("=" * 70)

print(f"\n{'Rank':<6} {'Score-based':<30} {'RRF':<30}")
print("-" * 70)
for i in range(max(len(results), len(rrf_results))):
    score_entry = results[i].chunk_id if i < len(results) else "(none)"
    rrf_entry = rrf_results[i].chunk_id if i < len(rrf_results) else "(none)"
    print(f"{i+1:<6} {score_entry:<30} {rrf_entry:<30}")

print("\nKey observation:")
print("  With RRF, chunks found by BOTH stores are boosted to the top,")
print("  regardless of their raw similarity scores. This produces more")
print("  robust rankings when combining heterogeneous retrieval sources.")

## Key Takeaways

1. **RAG = Retrieve + Augment + Generate.** It transforms an LLM from an unreliable knowledge store into a reliable reading comprehension engine by grounding answers in retrieved documents.

2. The **HybridRetriever** queries multiple vector stores and merges their results. This provides higher recall than any single store alone, because different stores (and index types) find different sets of approximate neighbors.

3. **Deduplication by chunk_id** prevents the same passage from appearing multiple times when it exists in more than one store.

4. **Reciprocal Rank Fusion (RRF)** is the recommended re-ranking strategy when combining results from heterogeneous stores. It uses rank position instead of raw scores, making it robust to score scale differences between stores.

5. The `RAGPipeline` class wires everything together: `DocumentProcessor` -> `Chunker` -> `EmbeddingModel` -> `VectorStore` -> `HybridRetriever` -> `LLM`. A single call to `pipeline.query("your question")` runs the full flow.

6. The prompt template explicitly instructs the LLM to **only use provided context** and to **say "I don't know"** when the answer is not present. This is the single most effective technique for reducing hallucination in RAG systems.

## Next Steps

* Install Ollama (`https://ollama.com/`) and pull a model (`ollama pull llama3.2`) to run the full generation pipeline.
* Try `RAGPipeline` end-to-end by ingesting your own documents and asking questions.
* Experiment with different chunking strategies and chunk sizes -- this is the highest-leverage tuning knob for RAG quality.
* Read the original RAG paper: "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (Lewis et al., 2020) -- https://arxiv.org/abs/2005.11401